In [ ]:
"""
Analisi Cu-64 (decadimenti nel nucleo): microdosimetria -> MKM -> LQ -> S(N), RBE(N)

Usage
-----
- Un file .root per ogni simulazione indipendente (seed diversi):  build/yz_run*.root
  -> l'errore statistico viene dalla dispersione tra i run.
- Se hai UN solo file, i "run" sono ottenuti ricampionando (bootstrap) gli eventi.

Il file si puo' usare a celle in VS Code/Jupyter (marcatori "# %%") oppure da terminale.
"""
# %% Import e parametri
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FILES = sorted(glob.glob("runs/run_*/yz.root")) or sorted(glob.glob("build/yz*.root"))
TREE = "yz"

# --- fisica / modello ---
R_d = 0.420        # um   raggio del dominio (= /microyz/det/Radius)
R_n = 4.0          # um   raggio del nucleo
y0 = 150.0         # keV/um  saturazione MKM
alpha_0 = 0.147    # Gy^-1
beta = 0.02        # Gy^-2
G = 1.0            # Lea-Catcheside: 1 = dose istantanea, <1 se protratta (riparazione)
rho = 1e-15        # kg/um^3 (acqua)
KEV_J = 1.602176634e-16
EV_J = 1.602176634e-19
S_TARGET = 0.10

# --- statistica ---
N_SIM = 400        # cellule virtuali per ogni valore di N
N_POINTS = 60      # punti sull'asse N
N_BOOT = 20        # pool ricampionati (solo se hai un file solo)
SEED = 12345

# schemi di peso per f(y): 1/n = correzione dentro l'evento, N/n = anche tra eventi
WEIGHT_MODES = ("1/n", "N/n")
PLOT_MODE = "N/n"

K_CONV = KEV_J / (rho * np.pi * R_d**2)   # Gy per keV/um  (~0.289)


# %% Lettura e controlli
def load(files):
    import uproot  # import qui cosi' le funzioni si possono testare senza uproot

    cols = ["y", "nbHits", "nbScoredHits", "Nuclear_dose"]
    return [uproot.open(f)[TREE].arrays(cols, library="pd") for f in files]


def check_alignment(df, label=""):
    """L'energia nel nucleo (da Nuclear_dose) deve essere >= energia nel dominio (da y)."""
    m_nucl = 1e3 * 4 / 3 * np.pi * (R_n * 1e-6) ** 3          # kg
    e_nucl = df["Nuclear_dose"].to_numpy() * m_nucl / EV_J     # eV
    eps = df["y"].to_numpy() * (4 * R_d / 3 * 1000.0)          # eV  (l = 4R/3)
    ok = np.mean(e_nucl >= eps * (1 - 1e-3) - 0.5)
    print(f"[check] {label or 'dati'}: righe con E_nucleo >= eps_dominio = {ok:.4f}  (deve essere 1.0000)")
    if ok < 0.999:
        print("        ATTENZIONE: dose e y probabilmente sfasate (fix EndOfEvent/AddNtupleRow non applicato?)")


# %% Microdosimetria
def weights(df, mode):
    n = df["nbScoredHits"].to_numpy(float)
    big_n = df["nbHits"].to_numpy(float)
    return 1.0 / n if mode == "1/n" else big_n / n


def micro(df, mode):
    y = df["y"].to_numpy()
    w = weights(df, mode)
    y_f = np.sum(w * y) / np.sum(w)
    y_d = np.sum(w * y**2) / np.sum(w * y)
    y_star = np.sum(w * y0**2 * (1.0 - np.exp(-(y / y0) ** 2))) / np.sum(w * y)
    return y_f, y_d, y_star


# %% Dosi e curve LQ
def sample_doses(dn, ns, rng):
    """D = somma di N dosi nucleari estratte (con reinserimento) per N_SIM cellule virtuali."""
    d_mat = np.empty((len(ns), N_SIM))
    for i, n in enumerate(ns):
        idx = rng.integers(0, dn.size, size=(N_SIM, int(n)), dtype=np.int32)
        d_mat[i] = dn[idx].sum(axis=1)
    return d_mat


def lq(d_mat, alpha):
    e = alpha * d_mat + beta * G * d_mat**2
    s = np.exp(-e)
    d_ref = (-alpha_0 + np.sqrt(alpha_0**2 + 4 * beta * e)) / (2 * beta)   # dose X acuta equivalente
    rbe = np.where(d_mat > 0, d_ref / np.where(d_mat > 0, d_mat, 1.0), np.nan)
    return s, rbe


def crossing(ns, s_mean, d_mean, target=S_TARGET):
    s = np.maximum(np.minimum.accumulate(s_mean), 1e-300)     # monotona decrescente
    log_s = np.log(s)
    n10 = np.interp(np.log(target), log_s[::-1], ns[::-1])
    d10 = np.interp(np.log(target), log_s[::-1], d_mean[::-1])
    return n10, d10


def d_ref_at(s_target):
    return (-alpha_0 + np.sqrt(alpha_0**2 - 4 * beta * np.log(s_target))) / (2 * beta)


# %% Analisi di un pool (= una simulazione)
def analyze_pool(df, ns, rng):
    dn = df["Nuclear_dose"].to_numpy()
    d_mat = sample_doses(dn, ns, rng)            # estratto una volta, riusato per tutti gli schemi di peso
    out = {}
    for mode in WEIGHT_MODES:
        y_f, y_d, y_star = micro(df, mode)
        alpha = alpha_0 + beta * K_CONV * y_star
        s, rbe = lq(d_mat, alpha)
        s_mean, d_mean = s.mean(axis=1), d_mat.mean(axis=1)
        n10, d10 = crossing(ns, s_mean, d_mean)
        out[mode] = dict(
            S=s_mean, S_cell=s.std(axis=1),
            rbe=np.nanmean(rbe, axis=1), rbe_cell=np.nanstd(rbe, axis=1),
            D=d_mean,
            yF=y_f, yD=y_d, ystar=y_star, alpha=alpha,
            N10=n10, D10=d10, rbe10=d_ref_at(S_TARGET) / d10,
            rbe_max=alpha / alpha_0,             # limite D -> 0
            z_per_decay=dn.mean(),
        )
    return out


def aggregate(results):
    """results: lista (un elemento per pool) di dict. Media e errore sulla media tra pool."""
    k = len(results)
    agg = {}
    for key in ("S", "rbe", "D"):
        arr = np.array([r[key] for r in results])
        agg[key] = arr.mean(axis=0)
        agg[key + "_sem"] = arr.std(axis=0, ddof=1) / np.sqrt(k)
    for key in ("S_cell", "rbe_cell"):
        agg[key] = np.mean([r[key] for r in results], axis=0)
    for key in ("yF", "yD", "ystar", "alpha", "N10", "D10", "rbe10", "rbe_max", "z_per_decay"):
        v = np.array([r[key] for r in results])
        agg[key] = v.mean()
        agg[key + "_sem"] = v.std(ddof=1) / np.sqrt(k)
    return agg


# %% Plot
def plot(ns, a):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 9), sharex=True)

    ax1.plot(ns, a["S"], color="#1f77b4", label=r"$\langle S\rangle$ Cu-64 (MKM-LQ)")
    ax1.fill_between(ns, np.clip(a["S"] - a["S_cell"], 1e-12, None), a["S"] + a["S_cell"],
                     color="#1f77b4", alpha=0.2, label="variabilità tra cellule (1σ)")
    ax1.errorbar(ns, a["S"], yerr=a["S_sem"], fmt="none", ecolor="k", elinewidth=0.8, capsize=2,
                 label="errore sulla media (tra simulazioni)")
    s_ref = np.exp(-alpha_0 * a["D"] - beta * a["D"] ** 2)
    ax1.plot(ns, s_ref, "--", color="gray", label="raggi X acuti, stessa dose media")
    ax1.axhline(S_TARGET, color="darkgreen", ls="--", lw=1.2)
    ax1.axvline(a["N10"], color="darkgray", ls=":", lw=1.2,
                label=f"$N_{{10}}$ = {a['N10']:.0f} ± {a['N10_sem']:.0f}")
    ax1.set_yscale("log")
    ax1.set_ylim(1e-3, 1.5)
    ax1.set_ylabel("Frazione di sopravvivenza S")
    ax1.grid(True, which="both", alpha=0.3)
    ax1.legend(loc="lower left", fontsize=9)

    ax2.plot(ns, a["rbe"], color="crimson", label="RBE")
    ax2.fill_between(ns, a["rbe"] - a["rbe_cell"], a["rbe"] + a["rbe_cell"], color="crimson", alpha=0.2)
    ax2.errorbar(ns, a["rbe"], yerr=a["rbe_sem"], fmt="none", ecolor="k", elinewidth=0.8, capsize=2)
    ax2.axhline(1.0, color="gray", ls="--", lw=1)
    ax2.set_xlabel("Numero di decadimenti N")
    ax2.set_ylabel("RBE")
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc="upper right", fontsize=9)

    plt.tight_layout()
    plt.savefig("cu64_S_RBE.png", dpi=200)
    plt.show()


def print_summary(agg_by_mode):
    print("\n" + "=" * 100)
    print(f"{'peso':<6}{'y_F':>12}{'y_D':>12}{'y*':>12}{'alpha':>14}{'RBE(D->0)':>12}"
          f"{'N10':>16}{'D10 [Gy]':>16}{'RBE10':>14}")
    print("-" * 100)
    for mode, a in agg_by_mode.items():
        print(f"{mode:<6}{a['yF']:>12.4f}{a['yD']:>12.4f}{a['ystar']:>12.4f}"
              f"{a['alpha']:>14.5f}{a['rbe_max']:>12.4f}"
              f"{a['N10']:>10.0f}±{a['N10_sem']:<5.0f}"
              f"{a['D10']:>10.4f}±{a['D10_sem']:<5.4f}"
              f"{a['rbe10']:>8.4f}±{a['rbe10_sem']:<5.4f}")
    print("=" * 100)
    print(f"Dose X di riferimento per S=10%: {d_ref_at(S_TARGET):.4f} Gy")
    print("Gli ± sono l'errore sulla media tra simulazioni indipendenti (o pool bootstrap).\n")


# %% Main
def make_pools(dfs, rng):
    if len(dfs) > 1:
        return dfs
    df = dfs[0]
    print(f"[info] un solo file: uso {N_BOOT} pool bootstrap degli stessi {len(df)} eventi")
    return [df.iloc[rng.integers(0, len(df), len(df))].reset_index(drop=True) for _ in range(N_BOOT)]


def run(dfs, labels=None):
    rng = np.random.default_rng(SEED)
    for i, d in enumerate(dfs):
        check_alignment(d, labels[i] if labels else f"pool {i}")

    if len(dfs) > 1:
        heads = [d["y"].to_numpy()[:200].tobytes() for d in dfs]
        if len(set(heads)) < len(heads):
            print("[ATTENZIONE] alcuni run hanno gli stessi eventi: i seed non sono cambiati! "
                  "L'errore tra run sarebbe sottostimato.")

    pools = make_pools(dfs, rng)

    # griglia N comune: fino a ~3x il N atteso per S=10% (stima con alpha = alpha_0)
    dn_mean = np.mean([p["Nuclear_dose"].mean() for p in pools])
    n_max = int(3 * d_ref_at(S_TARGET) / dn_mean)
    ns = np.unique(np.linspace(1, n_max, N_POINTS).astype(int))
    print(f"[info] dose nucleare media per decadimento = {dn_mean:.3e} Gy ; N da 1 a {n_max}")

    per_pool = [analyze_pool(p, ns, rng) for p in pools]
    agg_by_mode = {m: aggregate([r[m] for r in per_pool]) for m in WEIGHT_MODES}
    print_summary(agg_by_mode)
    plot(ns, agg_by_mode[PLOT_MODE])
    return ns, agg_by_mode


if __name__ == "__main__":
    if not FILES:
        raise SystemExit("Nessun file trovato: controlla FILES (es. build/yz*.root)")
    run(load(FILES), FILES)